# Comparative Evaluation: Fine-Tuned NLLB vs. SOTA LLMs

## Overview

This notebook represents the final benchmarking phase of the research pipeline. We compare our **Fine-Tuned NLLB-200 (LoRA)** model against current State-of-the-Art (SOTA) Large Language Models (LLMs) including **Gemini 2.5 Pro**, **GPT-5**, and **Claude Sonnet 4.5**.

The goal is to determine if a compact, parameter-efficient model fine-tuned on domain-specific data can compete with or outperform massive, general-purpose proprietary models in bidirectional **Odia (Low-Resource) $\leftrightarrow$ German (High-Resource)** translation.

## Evaluation Scope
1. **Models Evaluated:**
  * **NLLB-200 (LoRA):** Tested with both "Standard" (Beam=5) and "Optimized" (Beam=4, Penalty=0.6) inference strategies.
  * **Proprietary LLMs:** Google Gemini 2.5 Pro, OpenAI GPT-5, Anthropic Claude Sonnet 4.5.

2. **Metrics:**
  * **Lexical:** BLEU, chrF++, TER (via `sacrebleu`).
  * **Semantic:** COMET (Neural metric using Unbabel/wmt22-comet-da).
  
3. **Dataset:**
  * A hold-out test set (`test_bidirectional.jsonl`) processed into distinct directional subsets.
  
## Requirements
* **API Keys:** Valid keys for Google AI, OpenAI, and Anthropic stored in Colab Secrets.
* **Hardware:** A100 GPU (High-RAM) (Required for local LoRA inference and COMET metric calculation).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install -q transformers sacrebleu torch accelerate pandas bitsandbytes seaborn matplotlib peft unbabel-comet evaluate protobuf google-generativeai openai anthropic

In [ ]:
print("--- All Installed Packages (pip list) ---")
!pip list

--- All Installed Packages (pip list) ---
Package                                  Version
---------------------------------------- --------------------
absl-py                                  1.4.0
accelerate                               1.12.0
access                                   1.1.10.post3
affine                                   2.4.0
aiofiles                                 24.1.0
aiohappyeyeballs                         2.6.1
aiohttp                                  3.13.2
aiosignal                                1.4.0
aiosqlite                                0.22.0
alabaster                                1.0.0
albucore                                 0.0.24
albumentations                           2.0.8
ale-py                                   0.11.2
alembic                                  1.17.2
altair                                   5.5.0
annotated-doc                            0.0.4
annotated-types                          0.7.0
anthropic                         

In [ ]:
# import required libraries
import os
import json
import torch
import gc
import pandas as pd
import sacrebleu
import time
import random
from tqdm.auto import tqdm
from datasets import load_dataset
from evaluate import load as load_metric
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, BitsAndBytesConfig
from peft import PeftModel
import google.generativeai as genai
from openai import OpenAI
import anthropic
from google.colab import userdata

In [ ]:
# --- CONFIGURATION ---

# 1. DATA PATHS
DATA_DIR = "/content/drive/MyDrive/Research_Paper_Publication/data/transformed/"
TEST_FILE = os.path.join(DATA_DIR, "test_bidirectional.jsonl")
BASE_MODEL_NAME = "facebook/nllb-200-distilled-600M"
FINAL_LORA_PATH = "/content/drive/MyDrive/Research_Paper_Publication/model/lora-odia-german-translator-model-final-new-v1"
OUTPUT_CSV_PATH = "/content/drive/MyDrive/Research_Paper_Publication/eval/final_sota_comparison_full_testset.csv"

# --- API KEYS ---
try:
    GOOGLE_API_KEY = userdata.get('GOOGLE_AI_API_KEY')
    OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
    ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')

    genai.configure(api_key=GOOGLE_API_KEY)
    client_openai = OpenAI(api_key=OPENAI_API_KEY)
    client_anthropic = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
    print("✅ API Clients Configured.")
except Exception as e:
    print(f"⚠️ API Key Error: {e}. Please check Colab secrets.")

# --- SOTA MODEL VERSIONS ---
MODEL_GEMINI = "gemini-2.5-pro"
MODEL_GPT = "gpt-5"
MODEL_CLAUDE = "claude-sonnet-4-5"

print(f"Comparison Targets: {MODEL_GEMINI}, {MODEL_GPT}, {MODEL_CLAUDE}")

✅ API Clients Configured.
Comparison Targets: gemini-2.5-pro, gpt-5, claude-sonnet-4-5


In [ ]:
# ===================================
# 2. METRIC CALCULATION
# ===================================
def calculate_metrics(sources, preds, refs, model_name, direction):
    """
    Computes a comprehensive suite of evaluation metrics (BLEU, chrF++, TER, and COMET)
    for machine translation outputs.

    This function handles the specific formatting requirements for SacreBLEU (which expects
    references wrapped in a list) and manages the efficient loading and batching of the
    neural COMET metric to prevent memory errors.

    Args:
        sources (list of str): The source sentences (required for COMET metric).
        preds (list of str): The generated translations (hypotheses).
        refs (list of str): The ground truth reference translations.
        model_name (str): Identifier for the model being evaluated (e.g., "NLLB-200").
        direction (str): The translation direction label (e.g., "German -> Odia").

    Returns:
        dict: A dictionary containing the evaluation results:
            - 'Direction': Translation direction.
            - 'Model': Model name.
            - 'BLEU': Corpus-level BLEU score (0-100).
            - 'chrF++': Character n-gram F-score (0-100).
            - 'TER': Translation Edit Rate (0-100, lower is better).
            - 'COMET': Neural semantic similarity score (scaled to 0-100).
    """
    print(f"📈 Calculating metrics for {model_name} ({direction})...")

    # Clean predictions: Ensure all outputs are strings and remove leading/trailing whitespace.
    clean_preds = [str(p).strip() if p else "" for p in preds]

    # --- SACREBLEU FORMAT ---
    # SacreBLEU expects a list of reference lists (one list per reference file) to support
    # multi-reference evaluation (e.g., [[ref1_a, ref2_a...], [ref1_b...]]).
    # Since we have a single reference per sentence, we must wrap the list of strings
    # inside another list: [ ['ref1', 'ref2', ...] ].
    ref_formatted = [refs]

    # 1. SacreBLEU Metrics
    # BLEU: Standard n-gram precision metric.
    bleu = sacrebleu.BLEU().corpus_score(clean_preds, ref_formatted).score

    # chrF++: Character n-gram F-score with word order information (word_order=2).
    # This is often more robust for morphologically rich or low-resource languages.
    chrf = sacrebleu.CHRF(word_order=2).corpus_score(clean_preds, ref_formatted).score

    # TER: Translation Edit Rate. Measures the number of edits to match the reference.
    # Note: Unlike BLEU/chrF, a LOWER TER score is better.
    ter = sacrebleu.TER().corpus_score(clean_preds, ref_formatted).score

    # 2. COMET Metric
    # COMET (Crosslingual Optimized Metric for Evaluation of Translation) is a neural metric
    # that uses the Source, Hypothesis, and Reference. It correlates better with human judgment.
    comet_score = 0.0
    try:
        global comet_model
        # Lazy loading: Only load the heavy COMET model once to save time across multiple calls.
        if 'comet_model' not in globals():
            from comet import download_model, load_from_checkpoint
            # print("   Loading COMET model...")
            # Download the standard WMT22 model (top-performing reference-based metric)
            model_path = download_model("Unbabel/wmt22-comet-da")
            comet_model = load_from_checkpoint(model_path)

            # Move to GPU if available for faster inference
            if torch.cuda.is_available(): comet_model = comet_model.cuda()
            comet_model.eval()

        # Prepare data packet for COMET
        data = [{"src": s, "mt": m, "ref": r} for s, m, r in zip(sources, clean_preds, refs)]

        # Run inference in small batches (size=8) to avoid Out-Of-Memory (OOM) errors on consumer GPUs.
        comet_out = comet_model.predict(data,
                                        batch_size=8,
                                        gpus=1 if torch.cuda.is_available() else 0,
                                        progress_bar=False)

        # Convert system score to percentage scale (0-100) for consistency with BLEU.
        comet_score = comet_out.system_score * 100

    except Exception as e:
        print(f"   ⚠️ COMET skipped: {e}")

    return {
        "Direction": direction,
        "Model": model_name,
        "BLEU": bleu,
        "chrF++": chrf,
        "TER": ter,
        "COMET": comet_score
    }

In [ ]:
# ==========================================
# 3. HELPER: LLM API CALLS
# ==========================================
def get_llm_translation(model_name, prompt, max_retries=3):
    """
    Fetches a translation from a specified Large Language Model (LLM) API with retry logic.

    This function serves as a unified interface to interact with Google's Gemini,
    OpenAI's GPT, and Anthropic's Claude models. It handles authentication (via global clients),
    formats requests according to each provider's SDK, and implements a basic exponential
    backoff/retry mechanism for robustness against transient network errors or rate limits.

    Args:
        model_name (str): The specific model identifier (e.g., "gemini-1.5-pro", "gpt-4", "claude-3-sonnet").
                          The string must contain "gemini", "gpt", or "claude" to route correctly.
        prompt (str): The full input prompt string to send to the model.
        max_retries (int, optional): The maximum number of API call attempts before giving up.
                                     Defaults to 3.

    Returns:
        str: The generated text response (translation) from the model, stripped of leading/trailing whitespace.
             Returns an empty string "" if all retries fail.
    """
    # Attempt to fetch the response up to 'max_retries' times
    for attempt in range(max_retries):
        try:
            # --- Google Gemini Handling ---
            if "gemini" in model_name:
                # Initialize the specific GenerativeModel requested
                model = genai.GenerativeModel(model_name)
                # Generate content (default parameters are used here)
                response = model.generate_content(prompt)
                return response.text.strip()

            # --- OpenAI GPT Handling ---
            elif "gpt" in model_name:
                response = client_openai.chat.completions.create(
                    model=model_name,
                    messages=[{"role": "user", "content": prompt}],
                    temperature=1
                )
                # Extract content from the first choice
                return response.choices[0].message.content.strip()

            # --- Anthropic Claude Handling ---
            elif "claude" in model_name:
                msg = client_anthropic.messages.create(
                    model=model_name,
                    max_tokens=1024,
                    temperature=0,
                    messages=[{"role": "user", "content": prompt}]
                )
                # Extract text from the first content block
                return msg.content[0].text.strip()

        except Exception as e:
            # Error Handling:
            # If this was the final attempt, log the error explicitly so the user knows why it failed.
            if attempt == max_retries - 1:
                print(f"❌ API Error ({model_name}): {e}")

            # Wait for 1 second before retrying to respect rate limits and allow transient errors to clear.
            time.sleep(1)

    # Return empty string if all attempts fail to prevent downstream crashes
    return ""

In [ ]:
# ==========================================
# 4. MAIN EVALUATION LOOP
# ==========================================
def run_evaluation():
    """
    Orchestrates the complete benchmarking pipeline for the translation models.

    This function performs the following steps:
    1.  **Data Loading:** Loads the test dataset and splits it into the two translation directions
        (German -> Odia and Odia -> German).
    2.  **Model Loading:** Initializes the fine-tuned LoRA model with 4-bit quantization for
        memory-efficient inference.
    3.  **Iterative Evaluation:** Loops through both translation directions to evaluate:
        a.  **LoRA (Standard):** Using standard beam search parameters (Beam=5, Penalty=1.0).
        b.  **LoRA (Optimized):** Using parameters tuned via grid search (Beam=4, Penalty=0.6).
        c.  **SOTA LLMs:** Queries external APIs (Gemini, GPT, Claude) to get baseline comparisons.
    4.  **Metric Computation:** Calculates BLEU, chrF++, TER, and COMET for all models.
    5.  **Reporting:** Saves the aggregated results to a CSV file and displays a sorted summary table.
    """
    # 1. Load Dataset
    print("\n📊 Loading Test Data...")
    # 'load_dataset' handles JSONL parsing automatically.
    dataset = load_dataset("json", data_files={"test": TEST_FILE})["test"]

    # Split dataset based on the task prefix found in the input text.
    deu_ori_data = [ex for ex in dataset if ex["input_text"].startswith("translate German to Odia: ")]
    ori_deu_data = [ex for ex in dataset if ex["input_text"].startswith("translate Odia to German: ")]

    # 2. Load LoRA Model (4-bit)
    print("🔄 Loading LoRA Model...")
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)

    # Configure 4-bit quantization (NF4) to fit the 600M model + adapters into limited VRAM.
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.bfloat16
    )

    # Load base model frozen
    base_model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL_NAME,
                                                       quantization_config=bnb_config,
                                                       device_map="auto")

    # Attach trained adapters
    lora_model = PeftModel.from_pretrained(base_model, FINAL_LORA_PATH)
    lora_model.eval()   # Switch to eval mode (disables dropout, etc.)

    final_results = []

    # Define the evaluation tasks: Direction Name, Data Subset, Source Lang Code, Target Lang Code, Prefix
    directions = [
        ("German -> Odia", deu_ori_data, "deu_Latn", "ory_Orya", "translate German to Odia: "),
        ("Odia -> German", ori_deu_data, "ory_Orya", "deu_Latn", "translate Odia to German: ")
    ]

    for direction_name, data_subset, src_lang, tgt_lang, prefix in directions:
        print(f"\n🚀 --- {direction_name} ---")

        # Preprocess: strip prefixes for cleaner metric calculation later (especially for COMET)
        sources = [ex["input_text"].replace(prefix, "") for ex in data_subset]
        references = [ex["target_text"] for ex in data_subset]

        # Set tokenizer language tokens for correct generation
        tokenizer.src_lang = src_lang
        tokenizer.tgt_lang = tgt_lang
        forced_bos_token_id = tokenizer.convert_tokens_to_ids(tgt_lang)

        # ---------------------------------------------------------
        # A. LoRA Inference 1: BALANCED / STANDARD (Beams=5, Pen=1.0)
        # ---------------------------------------------------------
        # This setup represents the standard configuration for NMT tasks.
        print(f"   Running LoRA (Beams=5, Pen=1.0)...")
        lora_preds_balanced = []
        batch_size = 16

        # Process in batches
        for i in tqdm(range(0, len(sources), batch_size)):
            batch_src = sources[i:i+batch_size]
            # Tokenize and move to GPU
            inputs = tokenizer([prefix + s for s in batch_src], return_tensors="pt", padding=True, truncation=True).to(lora_model.device)

            with torch.no_grad():
                gen_tokens = lora_model.generate(
                    **inputs,
                    forced_bos_token_id=forced_bos_token_id,
                    max_length=1200,
                    num_beams=5,         # Standard NMT width
                    length_penalty=1.0   # Neutral length bias
                )
            lora_preds_balanced.extend(tokenizer.batch_decode(gen_tokens, skip_special_tokens=True))

        final_results.append(calculate_metrics(sources, lora_preds_balanced, references, "LoRA (Standard Inference)", direction_name))

        # ---------------------------------------------------------
        # B. LoRA Inference 2: OPTIMIZED (Beams=4, Pen=0.6)
        # ---------------------------------------------------------
        print(f"   Running LoRA (Beams=4, Pen=0.6)...")
        lora_preds_optimized = []
        for i in tqdm(range(0, len(sources), batch_size)):
            batch_src = sources[i:i+batch_size]
            inputs = tokenizer([prefix + s for s in batch_src], return_tensors="pt", padding=True, truncation=True).to(lora_model.device)
            with torch.no_grad():
                gen_tokens = lora_model.generate(
                    **inputs,
                    forced_bos_token_id=forced_bos_token_id,
                    max_length=1200,
                    num_beams=4,         # From Grid Search
                    length_penalty=0.6   # Favors shorter translations
                )
            lora_preds_optimized.extend(tokenizer.batch_decode(gen_tokens, skip_special_tokens=True))

        final_results.append(calculate_metrics(sources, lora_preds_optimized, references, "LoRA (Optimized Inference)", direction_name))

        # ---------------------------------------------------------
        # C. SOTA LLM Inference
        # ---------------------------------------------------------
        # Compare against massive proprietary models to establish a "David vs Goliath" baseline.
        llm_models = [
            ("Gemini 2.5 Pro", MODEL_GEMINI),
            ("GPT-5", MODEL_GPT),
            ("Claude Sonnet 4.5", MODEL_CLAUDE)
        ]

        for disp_name, model_id in llm_models:
            print(f"   Running {disp_name}...")
            llm_preds = []

            # Construct a clear system prompt to force the LLM into "Translator Mode"
            sys_prompt = f"Translate the following text from {src_lang.split('_')[0]} to {tgt_lang.split('_')[0]}. Output ONLY the translation."

            # Rate limit handling loop (APIs often have RPM limits)
            for src in tqdm(sources):
                llm_preds.append(get_llm_translation(model_id, f"{sys_prompt}\nText: {src}"))
                time.sleep(0.5) # Gentle rate limiting (adjust based on your API tier)

            final_results.append(calculate_metrics(sources, llm_preds, references, disp_name, direction_name))

    # 3. Save Results
    df = pd.DataFrame(final_results)
    df.to_csv(OUTPUT_CSV_PATH, index=False)
    print(f"\n✅ Done! Saved to {OUTPUT_CSV_PATH}")

    # 4. Display Final Table
    # Sort by Direction first, then by BLEU score (Descending) to show the winner at the top of each group.
    display(df.sort_values(by=["Direction", "BLEU"], ascending=[True, False]))

# --- RUN IT ---
# Execute the main evaluation function
run_evaluation()


📊 Loading Test Data...


Generating test split: 0 examples [00:00, ? examples/s]

🔄 Loading LoRA Model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

...

  0%|          | 0/368 [00:00<?, ?it/s]

📈 Calculating metrics for Claude Sonnet 4.5 (Odia -> German)...


INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]



✅ Done! Saved to /content/drive/MyDrive/Research_Paper_Publication/eval/final_sota_comparison_full_testset.csv


,Direction,Model,BLEU,chrF++,TER,COMET
0,German -> Odia,LoRA (Standard Inference),17.504817,46.465875,68.416419,82.312126
1,German -> Odia,LoRA (Optimized Inference),17.111851,46.174007,68.713547,82.221658
2,German -> Odia,Gemini 2.5 Pro,16.070632,46.295084,69.725982,83.606876
4,German -> Odia,Claude Sonnet 4.5,14.871904,44.830348,73.445582,82.608897
3,German -> Odia,GPT-5,8.073554,37.729930,80.103444,80.266220
7,Odia -> German,Gemini 2.5 Pro,33.654729,62.205780,58.810573,86.796387
9,Odia -> German,Claude Sonnet 4.5,32.025087,61.365033,60.394378,86.328290
5,Odia -> German,LoRA (Standard Inference),30.063955,56.019427,61.202014,81.950075
8,Odia -> German,GPT-5,29.923673,59.730667,64.086428,86.501465
6,Odia -> German,LoRA (Optimized Inference),29.610178,55.592106,61.569121,81.761593


### Observations

`1. The "Specialist vs. Generalist" Dynamic`:
The results reveal a distinct split in performance based on the target language's resource availability:
* **German $\rightarrow$ Odia (Low-Resource Target):** The **LoRA Fine-Tuned model (Standard Inference)** emerged as the clear winner with a **BLEU score of 17.50**, **chrF++ score of 46.46**, **TER score of 68.42**. It outperformed the strongest LLM, Gemini 2.5 Pro (16.07), and significantly surpassed GPT-5 (8.07). This confirms that for low-resource languages like Odia, a smaller, specialized model (600M parameters) fine-tuned on specific data often outperforms massive generalist LLMs.
* **Odia $\rightarrow$ German (High-Resource Target):** The massive LLMs demonstrated their superior priors for high-resource languages. **Gemini 2.5 Pro** took the lead with a **BLEU score of 33.65**, **chrF++ score of 62.20**, **TER score of 55.81** & **COMET score of 86.80**, followed closely by Claude Sonnet 4.5 (32.02, 61.36, 60.39, 86.32). The LoRA model trailed slightly at 30.06 BLEU, likely because the LLMs have seen vastly more German text during pre-training than the NLLB model.

`2. Metric Consistency & Nuance (BLEU, chrF++, TER vs. COMET)`:
* **German $\rightarrow$ Odia (Lexical Precision vs. Semantic Fluency):**
  * **Lexical Superiority (LoRA):** The LoRA model's dominance isn't limited to BLEU. It also achieved the highest **chrF++ score (46.47)** compared to Gemini 2.5 Pro (46.30). Since chrF++ is character-based, this indicates that the fine-tuned NLLB model is significantly better at handling the complex morphology and agglutinative nature of Odia words than the LLMs.
  * **Edit Distance (TER):** LoRA also achieved the best (lowest) **Translation Edit Rate (TER) of 68.42**, outperforming Gemini (69.73). This implies that LoRA's translations are structurally closer to the reference and would require fewer post-edits.
  * **The Divergence (COMET):** Despite losing on all lexical metrics (BLEU, chrF++, TER), **Gemini 2.5 Pro** achieved the highest **COMET score (83.61 vs. 82.31 for LoRA)**. This suggests that while LoRA is more precise in generating the exact Odia vocabulary and grammar (better for specific domain tasks), the LLM generates translations that are semantically richer or more fluent, even if they deviate structurally from the reference.
  
* **Odia $\rightarrow$ German (High-Resource Uniformity):**
  * In this direction, the metrics are entirely consistent. **Gemini 2.5 Pro** led across the board with the highest **chrF++ (62.21)** and the lowest **TER (58.81)**, reinforcing that for high-resource languages, massive pre-training yields superior performance in both morphology and semantics.

`3. Model Efficiency`:
* The LoRA model achieved SOTA-level performance in the difficult direction (De $\rightarrow$ Od) while running on a single GPU with 4-bit quantization. In contrast, the competing models (GPT-5, Gemini 2.5, Claude) require massive API infrastructure. This highlights the viability of the LoRA NLLB approach for efficient, on-premise deployment.